In [3]:
import sys
sys.path.append('../utilities/')
import pandas as pd
import numpy as np
from sklearn import svm
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import torch
from sentence_transformers import SentenceTransformer
from joblib import dump
from openai import OpenAI
from tqdm import tqdm
from mmd import MMD
import re
from sklearn.feature_extraction.text import CountVectorizer
import tiktoken
from collections import defaultdict, Counter
import os
from dotenv import load_dotenv
from nltk.corpus import stopwords
import nltk

# **Stopwords**

In [4]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(stop_words)

{"it's", 'her', 'yours', 'and', "we'd", "you're", "aren't", "he's", 'does', 'that', "we've", "hasn't", "they'd", "hadn't", 'are', 'they', "mightn't", "you've", 'during', 'too', 'doesn', 'just', 'shouldn', 'than', 'until', 'but', 'hadn', 'whom', 'few', 'll', 'you', 'your', 'so', 'weren', 'by', 'had', "mustn't", 'of', 'ma', 'once', 'how', 'under', 'can', 'then', 'were', 'is', 'aren', 'these', 'because', 'our', 'there', 'both', 'i', "they've", 'into', "doesn't", 'those', 'between', "shouldn't", 'below', 'hasn', 'what', 'again', 'ain', 'we', 'out', 'shan', 'was', "we're", 'about', 'above', "didn't", 'against', 'while', 'why', 'me', 'down', 'ours', "you'll", 'will', 'this', "don't", 'any', 'has', 'won', 't', 'such', 'should', 'wasn', 'do', 'theirs', 'or', 'very', 'from', 'not', 'y', 'his', 'some', 'my', "he'd", 'yourselves', "couldn't", 'be', "he'll", 'isn', 'didn', 'he', "needn't", 'himself', 'needn', 'being', 'no', "i've", 'don', 're', 'same', 'with', 'at', "they'll", 'a', 'him', 've', 'w

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/pranitgunjal/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# **Sentence Transformer**

In [5]:
sentence_transformer = SentenceTransformer('all-mpnet-base-v2')

# **Data Pre-processing**

In [6]:
train_df = pd.read_csv('../data/initial_datasets/dota2/dota2_train.csv')
test_df = pd.read_csv('../data/initial_datasets/dota2/dota2_test.csv')

In [7]:
train_df = train_df.sample(n=1000)

# **Tokenizer**

In [8]:
encoding = tiktoken.encoding_for_model("gpt-4")

In [9]:
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=API_KEY)

# **Get Co-Occurences**

In [10]:
train_df

,translated_message,label
2281,easy,0
1311,gg,0
184,"""boy churro the chaop""",0
2111,"""Yikes""",0
2311,XD,0
...,...,...
471,REPORT DAZZLE,0
1826,"""Really?>""",0
1403,"""friend, it turns out""",0
848,"""0 5""",0


In [10]:
text = train_df['translated_message'].to_list()

In [ ]:
tokens_list = []
for sentence in text:
    token_ids = encoding.encode(sentence)
    # Optionally, get string versions of tokens
    tokens = [encoding.decode([tid]) for tid in token_ids]
    tokens_list.append(tokens)

In [12]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [ ]:
top_k = 3

summary_text = ""
for token, counter in cooc.items():
    top = [w for w, _ in counter.most_common(top_k)]
    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"


In [16]:
tokens_list = []
for sentence in text:
    tokens = sentence.split()
    tokens_list.append(tokens)

In [17]:
cooc = defaultdict(Counter)

for tokens in tokens_list:
    unique_tokens = set(tokens)
    for token in unique_tokens:
        for other_token in unique_tokens:
            if token != other_token:
                cooc[token][other_token] += 1

In [22]:
top_k = 3
summary_text = ""

top_tokens = sorted(cooc.items(), key=lambda item: sum(item[1].values()), reverse=True)[:100]

for token, counter in top_tokens:
    if token in stop_words:
        continue

    top = [w for w, _ in counter.most_common() if w not in stop_words][:top_k]

    if not top:
        continue

    summary_text += f"'{token}' often appears with: {', '.join(top)}.\n"

In [23]:
print(summary_text)

'I' often appears with: love, feel, "and.
'The' often appears with: message, English., already.
'message' often appears with: The, English., already.
'"I' often appears with: idiot", reach,, dick".
'"you' often appears with: lane", will", 2.
'game' often appears with: 20, fucking, would.
'English.' often appears with: The, message, non-standard.
'random' often appears with: The, message, string.
'string' often appears with: The, random, message.
'appears' often appears with: The, random, message.
'going' often appears with: "he/she, there", "My.
'"and' often appears with: I, invoker?", fucked.
'YOU' often appears with: ARE, SUCH, "WHY.
'"What' often appears with: going, chat", decide.
'"what' often appears with: you,, dog", wonder".
'"how' often appears with: see", English:, type.
'fucked' often appears with: "you, lane", see.
'know' often appears with: shadow, use, "my.
'team' often appears with: "Gay, attacked", "my.
'time' often appears with: "Second, feeding", Mortred.
'like' often

In [25]:
instruction = (
    "You are a data generator tasked with creating realistic DOTA 2 chat messages. "
    "These chat messages should be labeled according to their sentiment: toxic or non-toxic.\n"
    "Base the style on typical video game chat messages — include informal internet language, typos, and abbreviations\n"
    "You will be given statistics about the distribution, including average chat length, standard deviation, and most common words associated with each label and their frequency.\n"
    "Generate exactly 10 realistic DOTA 2 chat messages, one per line.\n"
    "Each line should follow this format: the chat message in double quotes, followed by a space and then the label (0 for toxic, 1 for non-toxic).\n"
    "No extra formatting — just plain text output, one line per comment.\n"
    "Here is the format:\n"
    "\"gg dawg\" 0\n"
    "\"I hate u bitch\" 1"
)
input = (
    f"Here are the token co-occurences ordered by frequency:\n{summary_text}",
    f"Now, generate the 10 new comments below:"
)

In [26]:
response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
print(response.output_text)

"fucking game was trash" 0  
"The game was pretty fun" 1  
"You ARE SUCH an idiot" 0  
"Nice play, let's push lane" 1  
"I feel like we can win this" 1  
"stop feeding you noob" 0  
"great teamwork guys" 1  
"can't believe you picked that hero" 0  
"well played, gg team" 1  
"why you so bad at this game" 0


In [27]:
res = []
for i in tqdm(range(100)):
    response = client.responses.create(
        model="gpt-4o",
        instructions=instruction,
        input=input[0]
    )
    res.append(response.output_text)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [06:00<00:00,  3.61s/it]


In [28]:
labels = []
sentences = []
for i in range(100):
    for word in res[i].split("\n"):
        match = re.match(r'"(.*?)"\s*(-?\d+)', word)
        if match:
            quoted = match.group(1)      
            label = match.group(2)       
            sentences.append(quoted)
            labels.append(int(label))

In [29]:
generated_df = pd.DataFrame({
    'sentences': sentences,
    'labels': labels
})

In [32]:
second = generated_df

In [30]:
generated_df

,sentences,labels
0,that game was pure shit,0
1,gg wp team,1
2,fucking feeder report u!,0
3,"nice play, I see you improve",1
4,why u so bad lol,0
...,...,...
995,Why u gotta play like shit,0
996,I love how we're working together,1
997,"Stop going AFK, you're useless",0
998,Let's stick together and push mid,1


In [22]:
first = generated_df

In [36]:
combined = pd.concat([first, second, generated_df])

In [31]:
generated_df.to_csv('../data/generated/dota2/token_co_occurences/gen_token_co_occurences_no_stop_words.csv', index=False)